# 01. 원본 로드 · 컬럼 타입 분류 · 데이터 품질 점검

120,000행 × 64컬럼을 읽고, **타입을 자동 추론에 맡기지 않고 명시적으로 분류**한 뒤 결측·중복·고유값을 점검한다.

> 🐛 **여기서 첫 함정을 만난다.** 이름 키워드로 datetime 을 자동 추론하면 `First_time_iOS_by_vulnerable_user`(0/1 플래그)가 날짜로 오분류된다. 그래서 §1-1 에서 컬럼을 손으로 분류한다.

> **출처** — `원본/FDS_전처리_정리본.ipynb` (109셀 · Colab 실행본)
>
> 원본은 한 노트북에 §0~§15 를 전부 담고 있어 어디까지가 한 덩어리인지 알기 어려웠다.
> 이 저장소는 **절 경계 그대로** 5개로 나누고 **원본 실행 출력 98건을 모두 보존**했다.
> 코드는 손대지 않았다 — 셀 순서·내용 모두 원본과 동일하다.



### 재현 조건

| 항목 | 값 |
|---|---|
| 입력 | `train_final.csv` (원본 120,000행 × 64컬럼) |
| 원본 실행 경로 | `/content/train_final.csv` (Google Colab) |
| 저장소 경로 | `data/train.csv` — 용량(54MB) 때문에 미포함, `data/README.md` 참고 |
| 최종 산출물 | `X_tr/X_va/y_tr/y_va.parquet` (3차 전처리 · 58피처) + `label_encoders.pkl` · `le_target.pkl` |

> 노트북을 **순서대로(01→05)** 실행해야 한다. 앞 노트북의 `train` · `df` · `tr_idx`/`va_idx` 를
> 뒤 노트북이 이어받는 구조라, 단독 실행하면 `NameError` 가 난다.
> 각 노트북 첫 셀에 이어받는 변수를 명시해 두었다.

**변수 인계** — 이 노트북이 시작점이다. 만들어 내보내는 것: `train`(타입 변환본) · `train_raw`(문자열 원본 사본).

> 원본 셀 범위: `[0] ~ [11]` (총 12셀)


# 0. 원본 로드 및 검증

In [ ]:
import pandas as pd
import numpy as np
import re
import itertools


train = pd.read_csv('/content/train_final.csv', encoding='utf-8-sig')

# 0-2. 형식 검증용 원본 사본 (타입 변환 전 문자열 보존, 이후 수정 금지)
train_raw = train.copy()

print('Train shape:', train.shape)

Train shape: (120000, 64)


In [ ]:
# 0-3. Fraud_Type 분포
target_col = 'Fraud_Type'
print('Fraud_Type value counts')
display(train[target_col].value_counts().sort_index())
print('Fraud_Type ratio (%)')
display((train[target_col].value_counts(normalize=True).sort_index() * 100).round(4).rename('ratio_percent'))

Fraud_Type value counts


,count
Fraud_Type,
a,100
b,100
c,100
d,100
e,100
f,100
g,100
h,100
i,100


Fraud_Type ratio (%)


,ratio_percent
Fraud_Type,
a,0.0833
b,0.0833
c,0.0833
d,0.0833
e,0.0833
f,0.0833
g,0.0833
h,0.0833
i,0.0833


In [ ]:
# 0-4. 전체 중복 행
print('Train duplicated rows:', train.duplicated().sum())

Train duplicated rows: 0


# 1. 컬럼 타입 명시 분류 및 변환

자동 키워드 매칭은 `First_time_iOS_by_vulnerable_user`(0/1 플래그)를 이름만 보고 datetime으로 오분류한다.
이후 단계 전체의 안정성을 위해 **명세서 기준 명시 리스트**로 고정한다.

In [ ]:
# 1-1. 명시적 타입 분류

target_col = "Fraud_Type"

# 진짜 datetime 6종
datetime_cols = [
    "Customer_registration_datetime",
    "Account_creation_datetime",
    "Transaction_Datetime",
    "Last_atm_transaction_datetime",
    "Last_bank_branch_transaction_datetime",
    "Transaction_resumed_date",
]

timedelta_cols = ["Time_difference"]

# 수치형 후보
numeric_like_cols = train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# 0/1 플래그형 컬럼 자동 탐지
flag_cols = []
for col in numeric_like_cols:
    values = set(train[col].dropna().unique())
    if values.issubset({0, 1}):
        flag_cols.append(col)

# 플래그를 제외한 진짜 연속/금액/횟수 수치형
numeric_cols = [
    col for col in numeric_like_cols
    if col not in flag_cols
]

# 나머지는 범주형
categorical_cols = [
    col for col in train.columns
    if col not in numeric_cols
    + flag_cols
    + datetime_cols
    + timedelta_cols
    + [target_col]
]

# 오분류 방지: First_time_iOS_by_vulnerable_user는 플래그여야 함
assert "First_time_iOS_by_vulnerable_user" in flag_cols
assert "First_time_iOS_by_vulnerable_user" not in datetime_cols

print("datetime   :", len(datetime_cols), datetime_cols)
print("timedelta  :", len(timedelta_cols), timedelta_cols)
print("flag       :", len(flag_cols), flag_cols)
print("numeric    :", len(numeric_cols), numeric_cols)
print("categorical:", len(categorical_cols), categorical_cols)

total = (
    len(datetime_cols)
    + len(timedelta_cols)
    + len(flag_cols)
    + len(numeric_cols)
    + len(categorical_cols)
    + 1  # Fraud_Type
)

print("총 컬럼 수(train 기준, target 포함):", total)

datetime   : 6 ['Customer_registration_datetime', 'Account_creation_datetime', 'Transaction_Datetime', 'Last_atm_transaction_datetime', 'Last_bank_branch_transaction_datetime', 'Transaction_resumed_date']
timedelta  : 1 ['Time_difference']
flag       : 25 ['Customer_flag_change_of_authentication_1', 'Customer_flag_change_of_authentication_2', 'Customer_flag_change_of_authentication_3', 'Customer_flag_change_of_authentication_4', 'Customer_rooting_jailbreak_indicator', 'Customer_mobile_roaming_indicator', 'Customer_VPN_Indicator', 'Customer_flag_terminal_malicious_behavior_1', 'Customer_flag_terminal_malicious_behavior_2', 'Customer_flag_terminal_malicious_behavior_3', 'Customer_flag_terminal_malicious_behavior_4', 'Customer_flag_terminal_malicious_behavior_5', 'Customer_flag_terminal_malicious_behavior_6', 'Customer_inquery_atm_limit', 'Customer_increase_atm_limit', 'Account_indicator_release_limit_excess', 'Account_indicator_Openbanking', 'Account_release_suspention', 'Transaction_Fai

In [ ]:
# 1-2. 타입 변환
# datetime 6종 → datetime64
for col in datetime_cols:
    train[col] = pd.to_datetime(train[col], errors='coerce')

# Time_difference → timedelta → 초 단위 파생 (음수는 아직 보존: 아래 7단계에서 점검 후 처리)
train['Time_difference'] = pd.to_timedelta(train['Time_difference'], errors='coerce')
train['Time_difference_seconds'] = train['Time_difference'].dt.total_seconds()

print('Transaction_Datetime dtype:', train['Transaction_Datetime'].dtype)
print('First_time_iOS_by_vulnerable_user dtype (플래그 유지):', train['First_time_iOS_by_vulnerable_user'].dtype)
print('Time_difference_seconds 생성:', 'Time_difference_seconds' in train.columns)

Transaction_Datetime dtype: datetime64[ns]
First_time_iOS_by_vulnerable_user dtype (플래그 유지): int64
Time_difference_seconds 생성: True


In [ ]:
# 1-3. 타입 정리표
def role_of(col):
    if col == target_col: return 'target'
    if col in datetime_cols: return 'datetime'
    if col in timedelta_cols or col == 'Time_difference_seconds': return 'timedelta'
    if col in numeric_cols: return 'numeric'
    return 'categorical'

type_summary = pd.DataFrame({
    'column': train.columns,
    'dtype': train.dtypes.astype(str).values,
    'missing_count': train.isna().sum().values,
    'unique_count': train.nunique().values,
    'role': [role_of(c) for c in train.columns],
})
display(type_summary)

,column,dtype,missing_count,unique_count,role
0,ID,object,0,120000,categorical
1,Customer_Birthyear,int64,0,55,numeric
2,Customer_Gender,object,0,2,categorical
3,Customer_personal_identifier,object,0,1769,categorical
4,Customer_identification_number,object,0,4101,categorical
...,...,...,...,...,...
60,Transaction_history_with_the_account,int64,0,18,numeric
61,First_time_iOS_by_vulnerable_user,int64,0,2,categorical
62,Fraud_Type,object,0,13,target
63,Transaction_resumed_date,datetime64[ns],0,56200,datetime


# 2. 데이터 품질 점검

In [ ]:
# 2-1. 컬럼별 dtype / 결측 / 고유값
quality_overview = pd.DataFrame({
    'column': train.columns,
    'dtype': train.dtypes.astype(str).values,
    'missing_count': train.isna().sum().values,
    'missing_ratio': (train.isna().mean() * 100).round(4).values,
    'unique_count': train.nunique(dropna=False).values,
    'unique_ratio': (train.nunique(dropna=False) / len(train) * 100).round(4).values,
})
display(quality_overview)

,column,dtype,missing_count,missing_ratio,unique_count,unique_ratio
0,ID,object,0,0.0,120000,100.0000
1,Customer_Birthyear,int64,0,0.0,55,0.0458
2,Customer_Gender,object,0,0.0,2,0.0017
3,Customer_personal_identifier,object,0,0.0,1769,1.4742
4,Customer_identification_number,object,0,0.0,4101,3.4175
...,...,...,...,...,...,...
60,Transaction_history_with_the_account,int64,0,0.0,18,0.0150
61,First_time_iOS_by_vulnerable_user,int64,0,0.0,2,0.0017
62,Fraud_Type,object,0,0.0,13,0.0108
63,Transaction_resumed_date,datetime64[ns],0,0.0,56200,46.8333


In [ ]:
# 2-2. 결측 있는 컬럼만
missing_report = pd.DataFrame({
    'column': train.columns,
    'missing_count': train.isna().sum().values,
    'missing_ratio': (train.isna().mean() * 100).round(4).values,
})
display(missing_report[missing_report['missing_count'] > 0].sort_values('missing_count', ascending=False))
# 참고: Time_difference_seconds 의 결측(음수였던 값)은 아래 7단계 처리 후에 생긴다

,column,missing_count,missing_ratio


In [ ]:
# 2-3. 중복
print('Train duplicated rows :', train.duplicated().sum())
print('Train duplicated ratio:', round(train.duplicated().mean() * 100, 4), '%')

Train duplicated rows : 0
Train duplicated ratio: 0.0 %
